# Re-runs — replaying a finished run from any turn

`client.reruns` takes a finished run and starts a **new** one from a chosen turn, with the
conversation up to that point restored. Use it to reproduce a bad answer, try a prompt change
against a real conversation, or replay a production transcript against a new workflow version.

Six endpoints, in the order you would actually reach for them:

| Method | Does |
|---|---|
| `rerunnable_turns()` | Which turns can be re-run, and why not if they cannot |
| `preflight()` | What a re-run *would* carry over and drop. No side effects. |
| `create_token()` | Mint a token carrying the projected state |
| `preview_token()` | Show what the token restores, without starting it |
| `amend_token()` | Edit the transcript before replaying |
| `execute()` | Run it server-side and hand back the new run |

`execute()` is the shortcut — it does mint-and-run in one call. The token flow exists for when
you want to *look* at the projection first, or edit it.

> **⚠️ Only WebSocket-driven runs can be re-run.** The config snapshot re-running needs is
> written by `stream()`, not by `execute()`. A REST-driven run reports every turn as
> not-rerunnable with *"This run predates workflow config snapshots"* — a message that reads
> like the run is merely old. This notebook therefore drives its source run over `stream()`.

**See also:** [Re-runs guide](../docs/guides/reruns.md)

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

## 1. A source run, driven over WebSocket

Two nodes, no user input needed, so the run reaches a terminal state on its own.

In [ ]:
import _bootstrap  # noqa: F401

import interactly_configs as ic
from interactly import AsyncWorkflowClient, WorkflowCommand

client = AsyncWorkflowClient()

llm = ic.LLMGroupConfig(llms=[ic.OpenAILLMConfig(model=ic.OPENAIModel.GPT_4_1_MINI, max_tokens=80)])

greeting = ic.SayLLMNodeConfig(
    name="Greeting", is_start=True, wait_for_user_message=False, self_loop=False,
    main_response_config=ic.PromptConfig(prompt="Greet the caller in one short sentence."),
    llms_config=llm,
)
farewell = ic.SayStaticMessageNodeConfig(
    name="Farewell",
    static_messages_config=ic.StaticMessagesConfig(static_messages=["Goodbye."]),
)

config = ic.WorkflowConfigFullyHydrated(
    workflow_config=ic.WorkflowConfig(name="NB18: Re-runs"),
    node_configs=[greeting, farewell],
    edge_configs=[ic.DirectEdgeConfig(source_node_logical_id=greeting.logical_id,
                                      destination_node_logical_id=farewell.logical_id)],
)

workflow = await client.workflows.create_from_config(config, name="NB18: Re-runs")
WORKFLOW_ID = workflow.id
print("Created", WORKFLOW_ID)

In [ ]:
RUN_ID = None

async with client.runs.stream(workflow_id=WORKFLOW_ID, command=WorkflowCommand.START) as stream:
    async for event in stream:
        # The handshake frame carries the new run's id.
        if getattr(event, "workflow_run_id", None):
            RUN_ID = event.workflow_run_id
        if event.type == "assistant_response":
            print("🤖", event.output)
        if event.is_terminal():
            print(f"\n{event.type}")
            break

print("\nSource run:", RUN_ID)

## 2. Which turns can be re-run?

Ask before assuming. `rerunnable` is a server verdict, and `reason` tells you why when it is
`False` — the two common answers being "this run was not WebSocket-driven" and "the workflow has
changed incompatibly since".

`rerun_count` is how many times that turn has already been replayed.

In [ ]:
turns = await client.reruns.rerunnable_turns(RUN_ID)

for turn in turns.turns:
    mark = "✅" if turn.rerunnable else "❌"
    print(f"{mark} turn {turn.turn_index}  rerun_count={turn.rerun_count}  {turn.reason or ''}")

## 3. Preflight — what would carry over

`preflight()` has **no side effects**. It answers three things worth knowing before you commit:

- **`resolved_mode`** — `EXACT` when replaying against the same workflow and version. Replaying
  against a different target resolves to a looser mode, because node ids may not line up.
- **`node_matches`** — how each source node maps onto the target. `matched_by: "identity"` means
  the same logical id exists; a name match or no match at all is where a cross-workflow replay
  starts losing fidelity.
- **`suggested_entry_nodes`** — where the replay could start.

In [ ]:
pre = await client.reruns.preflight(RUN_ID, turn_index=0)

print(f"rerunnable   : {pre.rerunnable}")
print(f"resolved_mode: {pre.resolved_mode}")
print(f"is_resumable : {pre.is_resumable}  {pre.not_resumable_reason or ''}")
print(f"\nnode matches:")
for m in pre.node_matches:
    print(f"  {m.source_node_name:12s} → {m.target_node_name:12s}  by {m.matched_by}")
print(f"\nsuggested entry nodes: {len(pre.suggested_entry_nodes)}")

## 4. The token flow — look before you leap

`create_token()` mints a short-lived token carrying the projected state. Nothing runs yet.

`restored_message_count` is the useful number here: how much of the conversation the replay will
start with. `warnings` is where a lossy projection announces itself.

In [ ]:
token = await client.reruns.create_token(RUN_ID, turn_index=0)

print(f"token             : {token.rerun_token[:12]}…")
print(f"expires_in_seconds: {token.expires_in_seconds}")
print(f"resolved_mode     : {token.resolved_mode}")
print(f"restored_messages : {token.restored_message_count}")
print(f"warnings          : {token.warnings or 'none'}")

In [ ]:
preview = await client.reruns.preview_token(RUN_ID, 0, token.rerun_token)

print(f"effective_mode : {preview.effective_mode}   edited={preview.edited}")
print(f"resume thread  : {preview.resume_thread_id}")
print(f"removed msgs   : {preview.removed_message_count}\n")

# `threads` is deliberately List[Dict[str, Any]] rather than a typed model: it is a display
# payload whose shape follows the server, and typing it would make a newer server's extra
# field a validation error in an older SDK. Read it with .get().
for thread in preview.threads:
    print(f"thread {thread.get('thread_id')!r}:")
    for msg in thread.get("messages", []):
        editable = "✏️ " if msg.get("editable") else "🔒"
        print(f"  {editable} [{msg.get('index')}] {msg.get('sender', ''):9s} {(msg.get('text') or '')[:60]}")

### Editing the transcript before replaying

`amend_token()` rewrites the projection and returns the updated preview. The three edit lists —
`message_edits`, `message_appends`, `message_deletes` — are how you ask "what if the caller had
said something else?" without hand-building a run.

Not every message is editable: `editable` on each message says so, and `not_editable_reason`
explains why when it is `False`. A message carrying tool calls is the usual case — rewriting its
text would leave the tool results describing something that was never said.

You can also override `dynamic_variables` and `runtime_variables` here.

The cell below is commented out because this workflow's messages are short and there is nothing
interesting to change; uncomment it against a real transcript.

In [ ]:
# amended = await client.reruns.amend_token(
#     RUN_ID, 0, token.rerun_token,
#     message_edits=[{"thread_id": "0", "index": 1, "text": "Actually, I need to reschedule."}],
#     dynamic_variables={"member_id": "MBR-000001"},
# )
# print(f"edited={amended.edited}  removed={amended.removed_message_count}")

print("(see the commented cell above)")

### Redeeming a token yourself

A token is redeemed by passing it to `stream()`. This is the path to use when you want the
re-run's events live rather than fire-and-forget:

```python
async with client.runs.stream(workflow_id=WORKFLOW_ID, rerun_token=token.rerun_token) as stream:
    async for event in stream:
        ...
```

Two close codes matter here, and the SDK raises a specific exception for each rather than a
generic failure:

| Code | Exception | Meaning |
|---|---|---|
| 4006 | `InvalidStreamInputError` | The frame carried `initial_state`. A client is never the source of run state — use a token. |
| 4007 | `RerunTokenError` | The token expired, was minted for a different workflow or version, or the workflow changed since. |

Both subclass `StreamError`, so an existing `except StreamError` keeps working.

## 5. The shortcut: `execute()`

Mint and run in one call, server-side. Returns as soon as the run has **started** — `status` is
`"started"`, not a finished run — so follow the new `workflow_run_id` if you want the outcome.

In [ ]:
started = await client.reruns.execute(RUN_ID, turn_index=0)

print(f"new run       : {started.workflow_run_id}")
print(f"status        : {started.status}")
print(f"resolved_mode : {started.resolved_mode}")
print(f"warnings      : {started.warnings or 'none'}")

## 6. Finding re-runs later

Re-runs record their lineage, so you can ask "what came out of this run?" without keeping ids
yourself:

- `source_workflow_run_id` — every run re-run **from** this one;
- `source_turn_index` — narrow to one turn. It is **ignored on its own**: a turn index means
  nothing without the run it belongs to.

In [ ]:
page = await client.runs.list(source_workflow_run_id=RUN_ID)
print(f"{page.total} re-run(s) of {RUN_ID}")
async for run in page:
    print(f"  {run.id}  {run.status}")

narrowed = await client.runs.list(source_workflow_run_id=RUN_ID, source_turn_index=0)
print(f"\n{narrowed.total} from turn 0 specifically")

## Cleanup

In [ ]:
await client.workflows.delete(WORKFLOW_ID)
await client.close()
print("Deleted", WORKFLOW_ID)

## See also

- [`21_run_feedback.ipynb`](21_run_feedback.ipynb) — rating the runs worth re-running
- [`15_websocket_streaming_events.ipynb`](15_websocket_streaming_events.ipynb) — the driver that makes a run re-runnable
- [Re-runs guide](../docs/guides/reruns.md)

### Gotchas

- **Only WebSocket-driven runs can be re-run.** The snapshot comes from `stream()`.
- **`turn_index` is 0-based** — the position in `run.input_output_pairs`.
- **A token expires** (`expires_in_seconds`); mint a fresh one on `RerunTokenError`.
- **`execute()` returns `"started"`**, not a finished run.
- **`source_turn_index` alone does nothing** — pair it with `source_workflow_run_id`.